##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [9]:
#%pip install --no-compile "pinecone>=10.1.0" python-dotenv

From 'pinecone' library import Pinecone, ServerlessSpec.

In [10]:
from pinecone import Pinecone, ServerlessSpec, SchemaBuilder

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [11]:
import os, time
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "bootcamp-day9"
schema = (
    SchemaBuilder()
    .add_string_field("chunk_text", full_text_search={"language": "en"}, embed={})
    .build()
)
if not pc.indexes.exists(name=index_name):
    pc.indexes.create(name=index_name, schema=schema)

while not pc.indexes.describe(name=index_name).status.ready:
    time.sleep(2)

index = pc.index(name=index_name)

sentences_list = [
    "Dogs are loyal companions that love long walks in the park, playing fetch with their owners, and resting beside them every evening.",
    "My puppy gets very excited every morning when I grab his leash, because he knows we are going outside for a walk.",
    "Astronomers use powerful telescopes to study distant galaxies, hoping to understand how stars and planets formed billions of years ago in space.",
    "The rocket launched successfully this morning, carrying a new satellite into orbit to help scientists observe changes in the Earth climate.",
    "Regular exercise such as running, swimming or cycling strengthens the heart, improves mood and helps people stay healthy as they grow older.",
    "Eating plenty of fresh vegetables and fruit every day gives the body important vitamins and keeps the immune system strong and working well.",
    "Machine learning models learn patterns from large amounts of data and can then make predictions about new examples they have never seen.",
    "Neural networks are a type of artificial intelligence inspired by the human brain, made of layers of connected neurons that process information.",
    "Latvia is a small country on the Baltic Sea, known for its beautiful forests, sandy beaches and the historic old town of Riga.",
    "Every summer many tourists visit the Baltic coast to enjoy the beaches, eat local food and explore the charming small towns nearby.",
]

documents = [{"_id": str(i), "chunk_text": text} for i, text in enumerate(sentences_list)]
index.documents.upsert(namespace="__default__", documents=documents)

time.sleep(10)

query = "What is the best food for staying healthy?"
results = index.documents.search(
    namespace="__default__",
    top_k=3,
    score_by=[{"type": "embed", "fields": ["chunk_text"], "query": query}],
    include_fields=["chunk_text"],
)

Print search results.

In [12]:
print("Query:", query)
for match in results.matches:
    print(f"{match.score:.3f}  (id {match.id})  {match.chunk_text}")

Query: What is the best food for staying healthy?
0.449  (id 5)  Eating plenty of fresh vegetables and fruit every day gives the body important vitamins and keeps the immune system strong and working well.
0.313  (id 4)  Regular exercise such as running, swimming or cycling strengthens the heart, improves mood and helps people stay healthy as they grow older.
0.036  (id 9)  Every summer many tourists visit the Baltic coast to enjoy the beaches, eat local food and explore the charming small towns nearby.


In [13]:
pc.indexes.delete(name=index_name)